## N*N矩阵构造
### 采用Dijkstra 算法

In [1]:
import heapq
import numpy as np
import pandas as pd

#配置路径
graph = {
    1: {2: 3, 3: 4, 8: 1},
    2: {1: 3, 3: 2},
    3: {1: 4, 2: 2, 4: 2, 7: 5, 9: 2},
    4: {3: 2, 9: 1},
    5: {6: 2, 9: 2},
    6: {5: 2, 7: 1, 9: 3},
    7: {3: 5, 6: 1, 8: 2, 9: 4},
    8: {1: 1, 7: 2},
    9: {3: 2, 4: 1, 5: 2, 6: 3, 7: 4}
}

def dijkstra(graph, start_node):
    """
    单源最短路径：计算从 start_node 到图中所有其他节点的最短距离
    """
    # 初始化距离字典，所有节点默认距离为无穷大
    distances = {node: float('infinity') for node in graph}
    distances[start_node] = 0

    # 优先队列 (距离, 节点编号)，用于每次取出当前距离最小的节点
    priority_queue = [(0, start_node)]

    while priority_queue:
        # 取出当前距离最短的节点
        current_distance, current_node = heapq.heappop(priority_queue)

        # 如果取出的距离大于已记录的最短距离，直接跳过（说明是旧数据）
        if current_distance > distances[current_node]:
            continue

        # 遍历相邻节点，更新距离
        for neighbor, weight in graph[current_node].items():
            distance = current_distance + weight

            # 只有找到更短的路径时才更新并入队
            if distance < distances[neighbor]:
                distances[neighbor] = distance
                heapq.heappush(priority_queue, (distance, neighbor))

    return distances

def build_distance_matrix(graph, target_nodes=None):
    """
    构建 n * n 距离矩阵
    如果提供了 target_nodes，则只提取这些目标节点之间的子矩阵
    """
    all_nodes = sorted(list(graph.keys()))
    if target_nodes is None:
        target_nodes = all_nodes

    n = len(target_nodes)
    matrix = np.zeros((n, n))

    # 对每一个目标节点运行一次 Dijkstra
    for i, start in enumerate(target_nodes):
        # 得到从 start 到所有节点的最短距离
        shortest_paths = dijkstra(graph, start)

        # 将结果填入矩阵对应的行中
        for j, end in enumerate(target_nodes):
            matrix[i][j] = shortest_paths[end]

    return matrix, target_nodes

#全图距离矩阵
full_matrix, all_nodes = build_distance_matrix(graph)
print("=== 全图距离矩阵 ===")
df_full = pd.DataFrame(full_matrix, index=all_nodes, columns=all_nodes, dtype=int)
print(df_full)
print("\n")

# ==========================================
# 3. 提取目标景点的 n*n 降维矩阵 (宏观抽象)
# 假设图中标红的节点 (1, 3, 6, 9) 是我们真正要游玩的景点
# ==========================================
#red_nodes = [1, 3, 6, 9]
#target_matrix, t_nodes = build_distance_matrix(graph, target_nodes=red_nodes)
#print("=== 4x4 目标景点抽象矩阵 (仅包含节点 1, 3, 6, 9) ===")
#df_target = pd.DataFrame(target_matrix, index=t_nodes, columns=t_nodes, dtype=int)
#print(df_target)

=== 全图距离矩阵 ===
   1  2  3  4  5  6  7  8  9
1  0  3  4  6  6  4  3  1  6
2  3  0  2  4  6  7  6  4  4
3  4  2  0  2  4  5  5  5  2
4  6  4  2  0  3  4  5  7  1
5  6  6  4  3  0  2  3  5  2
6  4  7  5  4  2  0  1  3  3
7  3  6  5  5  3  1  0  2  4
8  1  4  5  7  5  3  2  0  6
9  6  4  2  1  2  3  4  6  0


